# RAG 入门指南

尽管大型语言模型（LLM）展现出强大的能力并支撑着许多高级应用场景，但它们也存在事实不一致和幻觉等问题。检索增强生成（RAG）是一种强大的方法，可以丰富 LLM 的能力并提高其可靠性。RAG  通过将 LLM 与外部知识相结合来实现这一目标，具体方式是在提示词中加入相关信息作为上下文，帮助完成特定任务。

本教程展示如何通过利用向量数据库和开源 LLM 来入门 RAG。为了展示 RAG 的强大功能，本用例将构建一个 RAG 系统，用于根据原始 ML 论文标题生成简短且易于阅读的论文标题。论文标题可能对普通读者来说过于专业，因此可以使用 RAG 基于先前创建的简短标题来生成简短标题，使科研论文标题更加通俗易懂，可用于科学传播，如Newsletter或博客文章。

在开始之前，让我们先安装所需的库：

In [ ]:
%%capture
!pip install chromadb tqdm fireworks-ai python-dotenv pandas
!pip install sentence-transformers

在继续之前，你需要获取一个 Fireworks API Key 来使用 Mistral 7B 模型。

获取 Fireworks API Key 的快速指南：https://readme.fireworks.ai/docs

In [2]:
import fireworks.client
import os
import dotenv
import chromadb
import json
from tqdm.auto import tqdm
import pandas as pd
import random

# 可以使用 Colab secrets 设置环境变量
dotenv.load_dotenv()

fireworks.client.api_key = os.getenv("FIREWORKS_API_KEY")

/home/fnliren/Downloads/miniconda3/envs/pe-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 入门指南

让我们定义一个函数来从 Fireworks 推理平台获取补全结果。

In [4]:
def get_completion(prompt, model=None, max_tokens=50):

    fw_model_dir = "accounts/fireworks/models/"

    if model is None:
        model = fw_model_dir + "gpt-oss-20b"
    else:
        model = fw_model_dir + model

    completion = fireworks.client.Completion.create(
        model=model,
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=0
    )

    return completion.choices[0].text

In [ ]:
def get_chat_completion(prompt, model=None, max_tokens=200):
    if model is None:
        model = "accounts/fireworks/models/gpt-oss-20b"
    
    # 转成 harmony 格式
    messages = [{"role": "user", "content": prompt}]
    
    completion = fireworks.client.ChatCompletion.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0
    )
    return completion.choices[0].message.content

让我们先用简单的提示词测试一下这个函数：

In [5]:
get_completion("Hello, my name is")

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7a5892235c50>


' John Doe. I am a software engineer with a passion for developing innovative solutions. I have experience in Java, Python, and JavaScript. I enjoy working in collaborative environments and am always eager to learn new technologies. In my free time, I love'

现在让我们用 Mistral-7B-Instruct 测试：

In [6]:
mistral_llm = "mistral-7b-instruct-4k"

get_completion("Hello, my name is")

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7a5891787c10>


' John Doe. I am a software engineer with a passion for developing innovative solutions. I have experience in Java, Python, and JavaScript. I enjoy working in collaborative environments and am always eager to learn new technologies. In my free time, I love'

Mistral 7B Instruct 模型需要使用特殊的指令标记 `[INST] <instruction> [/INST]` 来获得正确的行为。你可以在以下链接找到更多关于如何提示 Mistral 7B Instruct 的说明：https://docs.mistral.ai/llm/mistral-instruct-v0.1

In [25]:
get_completion("Tell me 2 jokes")

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x708af461a410>


' about the word "soup" and the ladle ladle?\n\nSure! Here are two jokes that involve the word "soup" and a ladle:\n\n1. Why did the ladle go to the soup kitchen? Because it heard'

In [26]:
get_chat_completion("Tell me 2 jokes")

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x708af44c7550>


'Sure thing! Here are two jokes for you:\n\n1. **Why don’t scientists trust atoms?**  \n   Because they make up everything!\n\n2. **What did the ocean say to the beach?**  \n   Nothing – it just waved.'

现在让我们用更复杂的包含指令的提示词试试：

In [11]:
prompt = """
Given the following wedding guest data, write a very short 3-sentences thank you letter:

{
  "name": "John Doe",
  "relationship": "Bride's cousin",
  "hometown": "New York, NY",
  "fun_fact": "Climbed Mount Everest in 2020",
  "attending_with": "Sophia Smith",
  "bride_groom_name": "Tom and Mary"
}

Use only the data provided in the JSON object above.

The senders of the letter is the bride and groom, Tom and Mary.
"""

get_chat_completion(prompt, max_tokens=2000)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x708c395c1e10>


"Dear John Doe, we are so grateful that you, our bride's cousin from New\u202fYork,\u202fNY, joined us on our special day. Your adventurous spirit, having climbed Mount Everest in 2020, added extra excitement to the celebration, especially as you came with Sophia Smith. Thank you for sharing this unforgettable moment with Tom and Mary."

## RAG 用例：生成简短论文标题

在 RAG 用例中，我们将使用一个包含每周热门 ML 论文列表的数据集。

用户将提供一个原始论文标题。然后我们将使用该数据集生成简短且吸引人的论文标题上下文，帮助为原始标题生成吸引人的标题。

### 第一步：加载数据集

首先让我们加载要使用的数据集：

In [12]:
# 从 data/ 文件夹加载数据集到 pandas DataFrame
# 数据集包含列名

ml_papers = pd.read_csv("./data/ml-potw-10232023.csv", header=0)

# 移除标题或描述为空的行
ml_papers = ml_papers.dropna(subset=["Title", "Description"])

In [13]:
ml_papers.head()

,Title,Description,PaperURL,TweetURL,Abstract
0,Llemma,an LLM for mathematics which is based on conti...,https://arxiv.org/abs/2310.10631,https://x.com/zhangir_azerbay/status/171409802...,"We present Llemma, a large language model for ..."
1,LLMs for Software Engineering,a comprehensive survey of LLMs for software en...,https://arxiv.org/abs/2310.03533,https://x.com/omarsar0/status/1713940983199506...,This paper provides a survey of the emerging a...
2,Self-RAG,presents a new retrieval-augmented framework t...,https://arxiv.org/abs/2310.11511,https://x.com/AkariAsai/status/171511027707796...,"Despite their remarkable capabilities, large l..."
3,Retrieval-Augmentation for Long-form Question ...,explores retrieval-augmented language models o...,https://arxiv.org/abs/2310.12150,https://x.com/omarsar0/status/1714986431859282...,We present a study of retrieval-augmented lang...
4,GenBench,presents a framework for characterizing and un...,https://www.nature.com/articles/s42256-023-007...,https://x.com/AIatMeta/status/1715041427283902...,NaN


In [14]:
# 将 DataFrame 转换为字典列表，只包含 Title 和 Description 列

ml_papers_dict = ml_papers.to_dict(orient="records")

In [15]:
ml_papers_dict[0]

{'Title': 'Llemma',
 'Description': 'an LLM for mathematics which is based on continued pretraining from Code Llama on the Proof-Pile-2 dataset; the dataset involves scientific paper, web data containing mathematics, and mathematical code; Llemma outperforms open base models and the unreleased Minerva on the MATH benchmark; the model is released, including dataset and code to replicate experiments.',
 'PaperURL': 'https://arxiv.org/abs/2310.10631',
 'TweetURL': 'https://x.com/zhangir_azerbay/status/1714098025956864031?s=20',
 'Abstract': 'We present Llemma, a large language model for mathematics. We continue pretraining Code Llama on the Proof-Pile-2, a mixture of scientific papers, web data containing mathematics, and mathematical code, yielding Llemma. On the MATH benchmark Llemma outperforms all known open base models, as well as the unreleased Minerva model suite on an equi-parameter basis. Moreover, Llemma is capable of tool use and formal theorem proving without any further finet

我们将使用 SentenceTransformer 生成嵌入向量，存储到 Chroma 文档数据库中。

In [21]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

class MyEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        batch_embeddings = embedding_model.encode(input)
        return batch_embeddings.tolist()

embed_fn = MyEmbeddingFunction()

# 初始化 chromadb 目录和客户端
client = chromadb.PersistentClient(path="./chromadb")

# 第一次：先删旧的，避免冲突
try:
    client.delete_collection(name="ml-papers-nov-2023")
except Exception:
    pass

# 创建 collection
collection = client.get_or_create_collection(
    name=f"ml-papers-nov-2023",
    embedding_function=embed_fn
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2050.13it/s]
/tmp/ipykernel_3486/610385386.py:10: DeprecationWarning: The class MyEmbeddingFunction does not implement __init__. This will be required in a future version.
  embed_fn = MyEmbeddingFunction()


现在让我们批量生成嵌入向量：

In [22]:
# 批量生成嵌入向量并索引标题
batch_size = 50

# 遍历批量数据，生成并存储嵌入向量
for i in tqdm(range(0, len(ml_papers_dict), batch_size)):

    i_end = min(i + batch_size, len(ml_papers_dict))
    batch = ml_papers_dict[i : i + batch_size]

    # 如果标题为空字符串则替换为 "No Title"
    batch_titles = [str(paper["Title"]) if str(paper["Title"]) != "" else "No Title" for paper in batch]
    batch_ids = [str(sum(ord(c) + random.randint(1, 10000) for c in paper["Title"])) for paper in batch]
    batch_metadata = [dict(url=paper["PaperURL"],
                           abstract=paper['Abstract'])
                           for paper in batch]

    # 生成嵌入向量
    batch_embeddings = embedding_model.encode(batch_titles)

    #  upsert 到 chromadb
    collection.upsert(
        ids=batch_ids,
        metadatas=batch_metadata,
        documents=batch_titles,
        embeddings=batch_embeddings.tolist(),
    )

100%|██████████| 9/9 [00:01<00:00,  4.65it/s]


现在我们可以测试检索器：

In [23]:
collection = client.get_or_create_collection(
    name=f"ml-papers-nov-2023",
    embedding_function=embed_fn
)

retriever_results = collection.query(
    query_texts=["Software Engineering"],
    n_results=2,
)

print(retriever_results["documents"])

[['LLMs for Software Engineering', 'Communicative Agents for Software Development']]


现在让我们组合最终的提示词：

In [24]:
# 用户查询
user_query = "S3Eval: A Synthetic, Scalable, Systematic Evaluation Suite for Large Language Models"

# 查询用户查询的相似结果
results = collection.query(
    query_texts=[user_query],
    n_results=10,
)

# 将标题连接成单个字符串
short_titles = '\n'.join(results['documents'][0])

prompt_template = f'''

Your main task is to generate 5 SUGGESTED_TITLES based for the PAPER_TITLE

You should mimic a similar style and length as SHORT_TITLES but PLEASE DO NOT include titles from SHORT_TITLES in the SUGGESTED_TITLES, only generate versions of the PAPER_TILE.

PAPER_TITLE: {user_query}

SHORT_TITLES: {short_titles}

SUGGESTED_TITLES:

'''

responses = get_chat_completion(prompt_template, max_tokens=2000)
suggested_titles = ''.join([str(r) for r in responses])

# 打印建议
print("Model Suggestions:")
print(suggested_titles)
print("\n\n\nPrompt Template:")
print(prompt_template)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x708af461b350>


Model Suggestions:
**SUGGESTED_TITLES:**

1. S3Eval: A Synthetic, Scalable, Systematic Benchmark for Large Language Models  
2. S3Eval: A Comprehensive Synthetic Evaluation Suite for LLMs  
3. S3Eval: Scalable Synthetic Testing for Large Language Models  
4. S3Eval: Systematic Synthetic Assessment of LLM Performance  
5. S3Eval: A Synthetic, Scalable Evaluation Framework for Large Language Models



Prompt Template:


Your main task is to generate 5 SUGGESTED_TITLES based for the PAPER_TITLE

You should mimic a similar style and length as SHORT_TITLES but PLEASE DO NOT include titles from SHORT_TITLES in the SUGGESTED_TITLES, only generate versions of the PAPER_TILE.

PAPER_TITLE: S3Eval: A Synthetic, Scalable, Systematic Evaluation Suite for Large Language Models

SHORT_TITLES: Pythia: A Suite for Analyzing Large Language Models Across Training and Scaling
ChemCrow: Augmenting large-language models with chemistry tools
A Survey of Large Language Models
LLaMA: Open and Efficient Founda

如你所见，LLM 生成的简短标题还算可以。这个用例仍然需要大量工作，可能还需要微调。出于本教程的目的，我们展示了一个使用 Fireworks 闪电般快速的模型进行 RAG 的简单应用。

在这里尝试其他开源模型：https://app.fireworks.ai/models

在此处阅读更多关于 Fireworks API 的信息：https://readme.fireworks.ai/reference/createchatcompletion